# Skin Cancer Classification using ResNet50

This notebook:
- Downloads HAM10000 dataset
- Preprocesses images
- Trains ResNet50
- Evaluates model accuracy
- Saves trained model
- Includes Streamlit deployment code


In [ ]:
# Install kagglehub if needed
# !pip install kagglehub


In [ ]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Dataset Path:", path)


In [ ]:
import pandas as pd

dataset_path = path

metadata_path = os.path.join(dataset_path, "HAM10000_metadata.csv")

df = pd.read_csv(metadata_path)

print(df.head())
print(df.shape)


In [ ]:
# Create image path dictionary

image_dir1 = os.path.join(dataset_path, "HAM10000_images_part_1")
image_dir2 = os.path.join(dataset_path, "HAM10000_images_part_2")

image_paths = {}

for dirname in [image_dir1, image_dir2]:
    for filename in os.listdir(dirname):
        image_id = filename.split(".")[0]
        image_paths[image_id] = os.path.join(dirname, filename)

df["path"] = df["image_id"].map(image_paths.get)

print(df["path"].head())


In [ ]:
# Use only Melanoma and Nevus classes

df = df[df["dx"].isin(["mel", "nv"])]

label_map = {
    "mel": 1,
    "nv": 0
}

df["label"] = df["dx"].map(label_map)

print(df["dx"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

# Better transforms with normalization + augmentation

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class SkinDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        img_path = self.dataframe.loc[idx, "path"]

        image = Image.open(img_path).convert("RGB")

        label = self.dataframe.loc[idx, "label"]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


In [ ]:
from torch.utils.data import DataLoader

train_dataset = SkinDataset(train_df, train_transform)
test_dataset = SkinDataset(test_df, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders Ready")


In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

print(device)


In [ ]:
import torch.optim as optim

# Weighted loss for class imbalance

class_weights = torch.tensor([1.0, 3.0]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)


In [ ]:
epochs = 10

best_accuracy = 0

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # Evaluation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {running_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.2f}%")

    # Save best model
    if accuracy > best_accuracy:

        best_accuracy = accuracy

        torch.save(
            model.state_dict(),
            "best_skin_cancer_model.pth"
        )

        print("Best model saved!")


In [ ]:
# Final model save

torch.save(model.state_dict(), "skin_cancer_model.pth")

print("Model Saved Successfully")


# Streamlit App Code

Save below code as `app.py`


In [ ]:
streamlit_code = '''

import streamlit as st
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Classes
classes = ["Nevus", "Melanoma"]

# Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load model
model = models.resnet50(weights=None)

model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(
    torch.load(
        "best_skin_cancer_model.pth",
        map_location=device
    )
)

model = model.to(device)

model.eval()

# Streamlit UI
st.title("Skin Cancer Detection")

uploaded_file = st.file_uploader(
    "Upload Skin Image",
    type=["jpg", "png", "jpeg"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file).convert("RGB")

    st.image(image, caption="Uploaded Image")

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(image_tensor)

        _, predicted = torch.max(outputs, 1)

    prediction = classes[predicted.item()]

    st.success(f"Prediction: {prediction}")

'''

print(streamlit_code)
